# Libraries

In [1]:
# Use this initial code to work in the notebook as if it were a module, that 
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath('.').split(os.sep + 'notebooks')[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
from pathlib import Path
import matplotlib.pyplot as plt

from spectralcrop import PathManager

# Load data

In [4]:
INTERIM_DATA = PathManager().get_abs_path_folder("interim")
FIGURES_FOLDER = PathManager().get_abs_path_folder("figures")
ZARR_PATH = INTERIM_DATA + "masked_reflectance.zarr"
REPORTS_DIR = INTERIM_DATA + "reports"

def to_numpy(arr):
    return arr.compute() if isinstance(arr.data, da.Array) else arr.values

In [5]:
ds = xr.open_zarr(ZARR_PATH, chunks={"band": 64, "y": 512, "x": 512})

# Variables clave (band,y,x) -> transponer a (y,x,band) para trabajar cómodo
refl_byx = ds["reflectance"].transpose("y", "x", "band")
wavelengths = ds["wavelength"].values.astype(float)
fwhm = ds["fwhm"].values.astype(float)
ndvi = ds["NDVI"]
veg_mask = ds["veg_mask"].astype("bool")
written_bands = ds["written_bands"]

print(refl_byx)
print("wavelengths:", wavelengths.shape, "nm  |  fwhm:", fwhm.shape, "nm")
print("NDVI dims:", ndvi.dims, "veg_mask dims:", veg_mask.dims)

# Consistencias útiles para checklist
wl_min, wl_max = float(np.nanmin(wavelengths)), float(np.nanmax(wavelengths))
b_out = int(refl_byx.sizes["band"])
wb_all_ones = bool(int(to_numpy(written_bands).min()) == 1 and int(to_numpy(written_bands).max()) == 1)
print(f"Bandas exportadas: {b_out}, λ: {wl_min:.1f}–{wl_max:.1f} nm, written_bands all=1? {wb_all_ones}")

C:\Users\JMONTOYA\AppData\Local\Temp\ipykernel_8848\144975827.py:1: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds = xr.open_zarr(ZARR_PATH, chunks={"band": 64, "y": 512, "x": 512})
C:\Users\JMONTOYA\AppData\Local\Temp\ipykernel_8848\144975827.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "band" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr(ZARR_PATH, chunks={"band": 64, "y": 512, "x": 512})


<xarray.DataArray 'reflectance' (y: 3660, x: 3438, band: 379)> Size: 19GB
dask.array<transpose, shape=(3660, 3438, 379), dtype=float32, chunksize=(512, 512, 64), chunktype=numpy.ndarray>
Coordinates:
  * band     (band) int32 2kB 0 1 2 3 4 5 6 7 ... 372 373 374 375 376 377 378
Dimensions without coordinates: y, x
Attributes:
    dimension_names:    ['band', 'y', 'x']
    _ARRAY_DIMENSIONS:  ['band', 'y', 'x']
wavelengths: (379,) nm  |  fwhm: (379,) nm
NDVI dims: ('y', 'x') veg_mask dims: ('y', 'x')
Bandas exportadas: 379, λ: 411.0–2449.2 nm, written_bands all=1? True


## Mover máscara no data hacia zarr principal (ahora está en interim/reports)

In [6]:
# Rutas (usando tu PathManager)
INTERIM_DIR = Path(PathManager().get_abs_path_folder("interim"))
ZARR_MAIN  = INTERIM_DIR / "masked_reflectance.zarr"
ZARR_MASK  = INTERIM_DIR / "reports" / "mask_nodata.zarr"

# Abrimos el Zarr principal con chunks compatibles
ds_main = xr.open_zarr(ZARR_MAIN, chunks={"band": 64, "y": 512, "x": 512})

# Abrimos el Zarr de máscara (puede traer un solo DataArray)
ds_mask = xr.open_zarr(ZARR_MASK)

print("Variables en mask_nodata.zarr:", list(ds_mask.data_vars))
print("Dims main:", ds_main.dims)

Variables en mask_nodata.zarr: ['mask_nodata']
Dims main: FrozenMappingWarningOnValuesAccess({'band': 379, 'y': 3660, 'x': 3438})


C:\Users\JMONTOYA\AppData\Local\Temp\ipykernel_8848\1178984255.py:7: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_main = xr.open_zarr(ZARR_MAIN, chunks={"band": 64, "y": 512, "x": 512})
C:\Users\JMONTOYA\AppData\Local\Temp\ipykernel_8848\1178984255.py:7: UserWarning: The specified chunks separate the stored chunks along dimension "band" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds_main = xr.open_zarr(ZARR_MAIN, chunks={"band": 64, "y": 512, "x": 512})


In [ ]:

ds_main

<xarray.Dataset> Size: 19GB
Dimensions:        (band: 379, y: 3660, x: 3438)
Coordinates:
  * band           (band) int32 2kB 0 1 2 3 4 5 6 ... 373 374 375 376 377 378
Dimensions without coordinates: y, x
Data variables:
    fwhm           (band) float32 2kB dask.array<chunksize=(64,), meta=np.ndarray>
    veg_mask       (y, x) uint8 13MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    NDVI           (y, x) float32 50MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    written_bands  (band) uint8 379B dask.array<chunksize=(64,), meta=np.ndarray>
    wavelength     (band) float32 2kB dask.array<chunksize=(64,), meta=np.ndarray>
    reflectance    (band, y, x) float32 19GB dask.array<chunksize=(64, 512, 512), meta=np.ndarray>
Attributes:
    description:            Reflectance cube (raw)
    units:                  unitless
    scale_applied:          10000.0
    clip_applied:           [0,1.2]
    nodata:                 NaN
    exclude_water_windows:  True
    water_windows_nm:       [[1340.0, 1440.0], [1800.0, 1950.0]]

In [8]:
ds_mask

<xarray.Dataset> Size: 13MB
Dimensions:      (y: 3660, x: 3438)
Dimensions without coordinates: y, x
Data variables:
    mask_nodata  (y, x) uint8 13MB dask.array<chunksize=(512, 512), meta=np.ndarray>

In [9]:

mask_var_name = next(iter(ds_mask.data_vars))
mask = ds_mask[mask_var_name]
print(mask_var_name)
mask

mask_nodata


<xarray.DataArray 'mask_nodata' (y: 3660, x: 3438)> Size: 13MB
dask.array<open_dataset-mask_nodata, shape=(3660, 3438), dtype=uint8, chunksize=(512, 512), chunktype=numpy.ndarray>
Dimensions without coordinates: y, x

In [14]:
# Asegurar que la máscara tenga dims (y, x) en ese orden
if set(mask.dims) != {"y", "x"}:
    # Si vienen otros dims (p.ej., ("rows","cols") o el orden inverso)
    # intenta transponer a (y,x)
    try:
        mask = mask.transpose("y", "x")
    except Exception as e:
        raise RuntimeError(f"No pude transponer la máscara a (y,x). Dims actuales: {mask.dims}") from e

mask

<xarray.DataArray 'mask_nodata' (y: 3660, x: 3438)> Size: 13MB
dask.array<open_dataset-mask_nodata, shape=(3660, 3438), dtype=uint8, chunksize=(512, 512), chunktype=numpy.ndarray>
Dimensions without coordinates: y, x

In [13]:



# Verificar tamaño y coord alignment con el Zarr principal
for dim in ("y", "x"):
    if dim not in ds_main.dims:
        raise RuntimeError(f"El Zarr principal no tiene dimensión '{dim}'. Dims: {ds_main.dims}")
    if ds_main.sizes[dim] != mask.sizes[dim]:
        raise RuntimeError(
            f"Desajuste espacial en '{dim}': main={ds_main.sizes[dim]} vs mask={mask.sizes[dim]}"
        )


In [15]:

# Revisar valores únicos (rápido con Dask; trae solo el histograma)
uniq = np.unique(mask.data.compute()) if isinstance(mask.data, da.Array) else np.unique(mask.values)
print("Valores únicos en la máscara:", uniq)


Valores únicos en la máscara: [0 1]


In [17]:
mask.values

array([[1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       ...,
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1]], shape=(3660, 3438), dtype=uint8)

In [18]:
# --- Escribir 'mask_nodata' en masked_reflectance.zarr ---
# Aseguramos chunking y tipo 'uint8' (compacto y claro)
mask_to_write = mask.astype("uint8").chunk({"y": 512, "x": 512}).rename("mask_nodata")
mask_to_write.attrs.update({
    "long_name": "NoData mask (0=valid, 1=NoData)",
    "source": str(ZARR_MASK),
})

# Escribimos agregando la variable al grupo existente
xr.Dataset({"mask_nodata": mask_to_write}).to_zarr(ZARR_MAIN, mode="a")
print("mask_nodata escrita en el Zarr principal:", ZARR_MAIN)

c:\Users\JMONTOYA\Documents\personal_projects\thesis\.venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


mask_nodata escrita en el Zarr principal: c:\Users\JMONTOYA\Documents\personal_projects\thesis\data\interim\masked_reflectance.zarr


In [ ]:
# Reconsolidar metadatos Zarr

import zarr

store = zarr.DirectoryStore(str(ZARR_MAIN))
zarr.consolidate_metadata(store)
print("Metadatos Zarr reconsolidados.")

AttributeError: module 'zarr' has no attribute 'DirectoryStore'

In [20]:
# Reabrir y verificar que la variable esté disponible y aliñada
ds_check = xr.open_zarr(ZARR_MAIN, chunks={"band": 64, "y": 512, "x": 512})
assert "mask_nodata" in ds_check.data_vars, "mask_nodata no se encontró en el Zarr principal."
print(ds_check["mask_nodata"])

<xarray.DataArray 'mask_nodata' (y: 3660, x: 3438)> Size: 13MB
dask.array<open_dataset-mask_nodata, shape=(3660, 3438), dtype=uint8, chunksize=(512, 512), chunktype=numpy.ndarray>
Dimensions without coordinates: y, x
Attributes:
    long_name:  NoData mask (0=valid, 1=NoData)
    source:     c:\Users\JMONTOYA\Documents\personal_projects\thesis\data\int...


C:\Users\JMONTOYA\AppData\Local\Temp\ipykernel_8848\606238354.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "band" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds_check = xr.open_zarr(ZARR_MAIN, chunks={"band": 64, "y": 512, "x": 512})
